# v47 系列NN — TCN+Attn(v29bの勝ち構成)を v47特徴(7バンク+gf, 781列)に載せ替え

v29b 実測: base 7.0346/blend 6.9382 → **combo(flip+prefix) 6.9403/blend 6.8711(★採用)** / full(+mtp) 6.8768。
本ノートはその **combo を既定**に、v47 の `artifacts_v47/data/train.parquet`(lag16・7バンク)+ gf特徴で再学習する。

- fold = tabular v47 と**完全同一** GroupKFold(5) → `gbm_v41_oof.npy` との正直ブレンド
- 使い方: cell1 の `RUN` を変えて上から再実行(`.pt` はプリセット別保存で skip 再開可)
- 最終セル = submit用パック書き出し(`submit_pack_nn.json`)→ `submit_nn_v47.ipynb` で提出


In [11]:
# ===== 設定 + プリセット選択 =====
import os, json, math, time, gc, copy, glob, re, warnings
import joblib
import numpy as np, pandas as pd
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore")

PROJECT = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
ART     = PROJECT / "v63" / "artifacts_v63"
PARQUET = ART / "data" / "train.parquet"
GF_NPY  = PROJECT / "v48" / "artifacts_v48" / "gf_feature.npy"   # gf 3列は parquet 外(npy)
DATA    = PROJECT / "rogii-wellbore-geology-prediction"
(ART / "nn").mkdir(parents=True, exist_ok=True)

# ---- プリセット(v29b と同一。★combo が v29b 実測ベスト) ----
PRESETS = {
    "base":     dict(flip=False, prefix=False, gbmch=False),
    "b_flip":   dict(flip=True,  prefix=False, gbmch=False),
    "d_prefix": dict(flip=False, prefix=True,  gbmch=False),
    "combo":    dict(flip=True,  prefix=True,  gbmch=False),   # ★v29b採用構成(blend 6.8711)
    "full":     dict(flip=True,  prefix=True,  gbmch=False, mtp=True),
}
RUN = "combo"
CFG = PRESETS[RUN]

N_SPLITS = 5
TOP_K    = 192
# v47 の列名(7バンク)に合わせた強制特徴: 正規表現で features.json から拾う
FORCE_PATTERNS = [
    r"pf_(ancc|self|nbr)(_[1-9])?",            # 各バンク絶対TVT(pf_ancc, pf_ancc_1..6, pf_self/nbr_1..7)
    r"pf_(self|nbr)_vs_tw(_[1-9])?",           # typewell PF との差
    r"pf_self_vs_nbr(_[1-9])?",
    r"pf_(self|nbr)_(mean|pstd)", r"pf_(self|nbr)_mean_vs_tw",
    r"pf_agree_std(_[1-9])?",
    r"pf_nbr_(has|nndist|nrefs)",
    r"md_since", r"z_grad",
    r"gf_(delta|std|vs_struct)",               # ★v48 grid filter 特徴
]
# --- モデル(v29b と同一) ---
CH = 192; KSIZE = 5; DILATIONS = [1, 2, 4, 8, 16, 32, 64, 128]; DROP = 0.10
USE_ATTN = True; ATTN_HEADS = 4; CH_DROP = 0.10
PRE_LEN = 512; PRE_C = 4; CTX_DIM = 64      # prefixエンコーダ
MTP_OFFS = [4, 16, 64]; MTP_W = 0.1
# --- 学習(v29b と同一) ---
HEADS = ["mdn2"]; SEEDS = [0, 1, 2]
EPOCHS = 60; LR = 3e-4; WD = 1e-4; BATCH_WELLS = 16; CROP = 1024
NOISE = 0.02; EMA_DECAY = 0.999; CLIP = 1.0; WARM_FRAC = 0.05
EVAL_TOK = 60_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pp_params = (json.loads((ART / "pp_params.json").read_text())
             if (ART / "pp_params.json").exists() else {"alpha": 1.0, "tau": 85, "w_pf": 0.09})
print(f"RUN={RUN} {CFG} | device={DEVICE} torch={torch.__version__}")


# ===== v63 pseudo(合成井戸)設定 =====
#   PSEUDO_MODE: off=ベースライン / pretrain=pseudoで事前学習→realでfinetune / da=realにpseudo井戸を混ぜる
PSEUDO_MODE = os.environ.get("PSEUDO_MODE", "pretrain")     # off | pretrain | da
PSEUDO_TAG  = os.environ.get("PSEUDO_TAG", "p1")
PSEUDO_PQ   = ART / "data" / f"pseudo_{PSEUDO_TAG}.parquet"
PSEUDO_DIR  = PROJECT / "v63" / "artifacts_v63" / f"pseudo_wells_{PSEUDO_TAG}"
PRE_EPOCHS  = int(os.environ.get("PRE_EPOCHS", "20"))  # pretrain: pseudoエポック数(flat LR)
DA_RATIO    = float(os.environ.get("DA_RATIO", "0.5")) # da: 1エポックに混ぜるpseudo井戸数の対train比
if PSEUDO_MODE != "off":
    RUN = f"{RUN}_{PSEUDO_MODE}-{PSEUDO_TAG}"          # ckpt/結果キーを分離(ベースラインを上書きしない)
    print(f"PSEUDO: mode={PSEUDO_MODE} tag={PSEUDO_TAG} pq={PSEUDO_PQ.name} pre_ep={PRE_EPOCHS} da_ratio={DA_RATIO}")


RUN=combo {'flip': True, 'prefix': True, 'gbmch': False} | device=cuda torch=2.11.0+cu128
PSEUDO: mode=pretrain tag=p1 pq=pseudo_p1.parquet pre_ep=20 da_ratio=0.5


In [12]:
# ===== 特徴選択(v47 tabular の LightGBM gain importance) =====
features_all = json.loads((ART / "features.json").read_text())   # 781列(gf含む, 学習時と同一)
rank_fp = ART / "feat_rank_nn.json"
if rank_fp.exists():
    rank = json.loads(rank_fp.read_text())
else:
    gain = np.zeros(len(features_all)); n_m = 0
    mroot = next((ART / m for m in ["models_fwd", "models_bwd", "models"] if (ART / m).exists()), None)
    for d in sorted(glob.glob(str(mroot / "lightgbm-*"))):
        for fp in sorted(glob.glob(str(Path(d) / "fold*.pkl"))):
            m = joblib.load(fp)
            gi = m.booster_.feature_importance(importance_type="gain")
            assert len(gi) == len(features_all), (len(gi), len(features_all))
            gain += gi; n_m += 1
    rank = [features_all[i] for i in np.argsort(-gain)]
    rank_fp.write_text(json.dumps(rank)); print("importance from", n_m, "lightgbm fold models")
rank = [c for c in rank if c != "last_known_tvt"]
_force_re = [re.compile(p + "$") for p in FORCE_PATTERNS]
feats_nn = [c for c in features_all if any(r.fullmatch(c) for r in _force_re)]
print("強制特徴:", len(feats_nn))
for c in rank:
    if len(feats_nn) >= TOP_K: break
    if c not in feats_nn: feats_nn.append(c)
print("NN特徴:", len(feats_nn), "| gf:", [c for c in feats_nn if c.startswith("gf_")])


強制特徴: 57
NN特徴: 192 | gf: ['gf_delta', 'gf_std', 'gf_vs_struct']


In [13]:
# ===== データ読込 + 前処理(v29bと同一) + prefix抽出 =====
import pyarrow.parquet as pq
_pq_names = set(pq.read_schema(PARQUET).names)
_gf_cols = [c for c in feats_nn if c not in _pq_names]           # gf_delta/gf_std/gf_vs_struct(npy由来)
need = list(dict.fromkeys(["well", "target", "last_known_tvt", "md_since", "pf_ancc"]
                          + [c for c in feats_nn if c in _pq_names]))
tbl = pq.read_table(PARQUET, columns=need)
N = tbl.num_rows
g      = tbl.column("well").to_pandas().to_numpy()
y      = tbl.column("target").to_numpy(zero_copy_only=False).astype(np.float64)
lk     = tbl.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
mds    = tbl.column("md_since").to_numpy(zero_copy_only=False).astype(np.float64)
pf_abs = tbl.column("pf_ancc").to_numpy(zero_copy_only=False).astype(np.float64)
X = np.empty((N, len(feats_nn)), np.float32)
_GF = np.load(GF_NPY) if _gf_cols else None
_gf_names = ["gf_delta", "gf_std", "gf_vs_struct"]
for j, c in enumerate(feats_nn):
    if c in _pq_names:
        X[:, j] = tbl.column(c).to_numpy(zero_copy_only=False).astype(np.float32)
    else:
        X[:, j] = np.nan_to_num(_GF[:N, _gf_names.index(c)], nan=0.0).astype(np.float32)
del tbl, _GF; gc.collect()

folds = list(GroupKFold(n_splits=N_SPLITS).split(np.zeros(N), y, groups=g))   # tabular v47 と同一

samp = np.arange(0, N, max(1, N // 120000))
abs2res = []
for j in range(X.shape[1]):
    v = X[samp, j].astype(np.float64); m = np.isfinite(v)
    ok = m.sum() > 100 and np.nanstd(v[m]) > 1e-9
    c_ = float(np.corrcoef(v[m], lk[samp][m])[0, 1]) if ok else 0.0
    a = bool(c_ > 0.98); abs2res.append(a)
    if a: X[:, j] = X[:, j] - lk.astype(np.float32)
print("絶対TVT→残差化:", int(sum(abs2res)), "列")

med = np.nanmedian(X, axis=0)
q75 = np.nanpercentile(X, 75, axis=0); q25 = np.nanpercentile(X, 25, axis=0)
scl = np.maximum((q75 - q25) / 1.349, 1e-6)
X = (X - med) / scl
X = np.clip(np.nan_to_num(X, nan=0.0, posinf=8.0, neginf=-8.0), -8.0, 8.0).astype(np.float32)

t_mean = float(y.mean()); t_scale = float(y.std())
yn = ((y - t_mean) / t_scale).astype(np.float32)

gbm_oof = np.load(ART / "gbm_v41_oof.npy").astype(np.float64)   # v47 tabular アンサンブルOOF(残差)
assert len(gbm_oof) == N, (len(gbm_oof), N)

wid_order = pd.unique(g)
well_rows = {}
for w in wid_order:
    idx = np.where(g == w)[0]
    well_rows[w] = idx[np.argsort(mds[idx], kind="stable")]
fold_wells = [(pd.unique(g[tr]), pd.unique(g[va])) for tr, va in folds]

# ---- 既知ゾーン(prefix)系列の抽出(キャッシュ) ----
PREF = {}
if CFG["prefix"]:
    pre_fp = ART / "prefix_cache_nn.npz"
    if pre_fp.exists():
        z = np.load(pre_fp, allow_pickle=False)
        PREF = {w: np.nan_to_num(z[w], nan=0.0, posinf=8.0, neginf=-8.0)
                for w in z.files if w in set(wid_order)}
        print("prefix cache load:", len(PREF), "wells")
    else:
        grs = []
        raw = {}
        for w in wid_order:
            f = DATA / "train" / f"{w}__horizontal_well.csv"
            if not f.exists(): continue
            df = pd.read_csv(f, usecols=["MD", "Z", "GR", "TVT_input"])
            kn = df.dropna(subset=["MD", "Z", "GR", "TVT_input"]).tail(PRE_LEN)
            if len(kn) < 8: continue
            raw[w] = kn.to_numpy(np.float64)   # [MD, Z, GR, TVT_input]
            grs.append(raw[w][:, 2])
        allgr = np.concatenate(grs)
        gmed = float(np.nanmedian(allgr))
        giqr = float(max(np.nanpercentile(allgr, 75) - np.nanpercentile(allgr, 25), 1e-6))
        for w, a in raw.items():
            gr_n = np.clip((a[:, 2] - gmed) / giqr, -8, 8)
            tvt_rel = (a[:, 3] - a[-1, 3]) / t_scale
            z_rel = (a[:, 1] - a[-1, 1]) / 50.0
            md_rel = (a[:, 0] - a[-1, 0]) / 1000.0
            PREF[w] = np.nan_to_num(np.stack([gr_n, tvt_rel, z_rel, md_rel], axis=1),
                                    nan=0.0, posinf=8.0, neginf=-8.0).astype(np.float32)
        np.savez_compressed(pre_fp, **PREF)
        print("prefix extracted:", len(PREF), "wells | gr med/iqr", round(gmed, 1), round(giqr, 1))

K_IN = X.shape[1]
print("rows", N, "| wells", len(wid_order), "| K", K_IN, "| t_mean/scale", round(t_mean, 3), "/", round(t_scale, 3))


# ===== v63 pseudo(合成井戸)読込: real統計で正規化し X/yn/well_rows に追記 =====
PS_WELLS = []; PS_FOLD = {}
if PSEUDO_MODE != "off":
    assert PSEUDO_PQ.exists(), f"pseudo parquet がない: {PSEUDO_PQ}(create_v63.py を PSEUDO_DIR 付きで実行)"
    _man = json.loads((PSEUDO_DIR / "manifest.json").read_text(encoding="utf-8"))
    _donor_of = {e["id"]: e["donor"] for e in _man["wells"]}
    _pqn = set(pq.read_schema(PSEUDO_PQ).names)
    _needp = list(dict.fromkeys(["well", "target", "last_known_tvt", "md_since"] + [c for c in feats_nn if c in _pqn]))
    _tblp = pq.read_table(PSEUDO_PQ, columns=_needp)
    Np = _tblp.num_rows
    gp   = _tblp.column("well").to_pandas().to_numpy()
    yp   = _tblp.column("target").to_numpy(zero_copy_only=False).astype(np.float64)
    lkp  = _tblp.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
    mdsp = _tblp.column("md_since").to_numpy(zero_copy_only=False).astype(np.float64)
    Xp = np.empty((Np, len(feats_nn)), np.float32)
    for j, c in enumerate(feats_nn):
        if c in _pqn:
            Xp[:, j] = _tblp.column(c).to_numpy(zero_copy_only=False).astype(np.float32)
        else:
            Xp[:, j] = np.nan                          # gf等のparquet外列 → 正規化後0(中央値相当)
    del _tblp; gc.collect()
    for j in range(Xp.shape[1]):                       # realと同じ残差化判定を適用
        if abs2res[j]: Xp[:, j] = Xp[:, j] - lkp.astype(np.float32)
    Xp = (Xp - med) / scl
    Xp = np.clip(np.nan_to_num(Xp, nan=0.0, posinf=8.0, neginf=-8.0), -8.0, 8.0).astype(np.float32)
    ynp = ((yp - t_mean) / t_scale).astype(np.float32)
    _N0 = len(yn)
    X = np.concatenate([X, Xp]); yn = np.concatenate([yn, ynp])
    _fold_of_well = {w: k_ for k_, (_tw_, _vw_) in enumerate(fold_wells) for w in _vw_}
    for w in pd.unique(gp):
        idx = np.where(gp == w)[0]
        well_rows[w] = (idx[np.argsort(mdsp[idx], kind="stable")] + _N0)
        PS_WELLS.append(w)
        PS_FOLD[w] = _fold_of_well.get(_donor_of.get(w, ""), -1)   # -1=donorがfold外→全foldで使用可
    # prefix系列(既知ゾーン)を pseudo dir から抽出(real統計 gmed/giqr で正規化)
    if CFG["prefix"]:
        _stats_fp = ART / "gr_stats_nn.json"
        if _stats_fp.exists():
            _st = json.loads(_stats_fp.read_text()); gmed_, giqr_ = _st["gmed"], _st["giqr"]
        else:
            _grs = []
            for w in wid_order:
                f = DATA / "train" / f"{w}__horizontal_well.csv"
                if not f.exists(): continue
                _df = pd.read_csv(f, usecols=["GR", "TVT_input"]).dropna()
                if len(_df) >= 8: _grs.append(_df["GR"].to_numpy(np.float64)[-PRE_LEN:])
            _a = np.concatenate(_grs)
            gmed_ = float(np.nanmedian(_a))
            giqr_ = float(max(np.nanpercentile(_a, 75) - np.nanpercentile(_a, 25), 1e-6))
            _stats_fp.write_text(json.dumps(dict(gmed=gmed_, giqr=giqr_)))
        for w in PS_WELLS:
            f = PSEUDO_DIR / f"{w}__horizontal_well.csv"
            if not f.exists(): continue
            _df = pd.read_csv(f, usecols=["MD", "Z", "GR", "TVT_input"])
            _kn = _df.dropna(subset=["MD", "Z", "GR", "TVT_input"]).tail(PRE_LEN)
            if len(_kn) < 8: continue
            _a = _kn.to_numpy(np.float64)
            PREF[w] = np.nan_to_num(np.stack([
                np.clip((_a[:, 2] - gmed_) / giqr_, -8, 8),
                (_a[:, 3] - _a[-1, 3]) / t_scale,
                (_a[:, 1] - _a[-1, 1]) / 50.0,
                (_a[:, 0] - _a[-1, 0]) / 1000.0], axis=1), nan=0.0, posinf=8.0, neginf=-8.0).astype(np.float32)
    print(f"pseudo: wells={len(PS_WELLS)} rows={Np} fold内訳:",
          {k_: sum(1 for w in PS_WELLS if PS_FOLD[w] == k_) for k_ in range(-1, N_SPLITS)})


絶対TVT→残差化: 24 列
prefix cache load: 773 wells
rows 3783989 | wells 773 | K 192 | t_mean/scale 1.596 / 15.83
pseudo: wells=300 rows=1473190 fold内訳: {-1: 0, 0: 64, 1: 60, 2: 60, 3: 62, 4: 54}


In [14]:
# ===== モデル定義(SeqNet + PrefixEnc) + ユーティリティ =====
class ChanLN(nn.Module):
    def __init__(self, c):
        super().__init__(); self.ln = nn.LayerNorm(c)
    def forward(self, x):
        return self.ln(x.transpose(1, 2)).transpose(1, 2)

class Block(nn.Module):
    def __init__(self, c, dil, k, drop):
        super().__init__()
        self.conv = nn.Conv1d(c, c, k, padding=dil * (k - 1) // 2, dilation=dil)
        self.ln = ChanLN(c); self.pw = nn.Conv1d(c, c, 1); self.do = nn.Dropout(drop)
    def forward(self, x, m):
        h = self.conv(x * m); h = F.gelu(h); h = self.ln(h); h = self.do(self.pw(h))
        return (x + h) * m

class SelfAttn(nn.Module):
    def __init__(self, ch, heads, drop):
        super().__init__()
        self.h = heads; self.dk = ch // heads; self.p = drop
        self.qkv = nn.Linear(ch, 3 * ch); self.proj = nn.Linear(ch, ch)
    def forward(self, x, kmask):
        B, L, C = x.shape
        q, k, v = self.qkv(x).chunk(3, -1)
        q = q.view(B, L, self.h, self.dk).transpose(1, 2)
        k = k.view(B, L, self.h, self.dk).transpose(1, 2)
        v = v.view(B, L, self.h, self.dk).transpose(1, 2)
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=kmask[:, None, None, :],
                                           dropout_p=self.p if self.training else 0.0)
        return self.proj(o.transpose(1, 2).reshape(B, L, C))

class PrefixEnc(nn.Module):
    """既知ゾーン系列 → 井戸コンテキストベクトル(masked mean+max pooling)。"""
    def __init__(self, c_in, ctx):
        super().__init__()
        self.c1 = nn.Conv1d(c_in, 64, 5, padding=2)
        self.c2 = nn.Conv1d(64, 64, 5, padding=8, dilation=4)
        self.fc = nn.Linear(128, ctx)
    def forward(self, x, m):  # x:(B,C,Lp), m:(B,1,Lp)
        h = F.gelu(self.c1(x * m)); h = F.gelu(self.c2(h)) * m
        s = m.sum(-1)                                   # (B,1)
        mean = h.sum(-1) / s.clamp(min=1.0)
        mx = (h + (m - 1.0) * 1e4).amax(-1)
        mx = torch.where(s > 0, mx, torch.zeros_like(mx))
        return self.fc(torch.cat([mean, mx], dim=-1)) * (s > 0).float()

class SeqNet(nn.Module):
    def __init__(self, k_in, head, ch, ksize, dilations, drop, use_attn, attn_heads, ch_drop,
                 use_ctx=False, pre_c=4, ctx_dim=64, use_mtp=False):
        super().__init__()
        self.head = head; self.use_ctx = use_ctx
        self.inp = nn.Conv1d(k_in, ch, 1); self.indrop = nn.Dropout1d(ch_drop)
        if use_ctx:
            self.pre = PrefixEnc(pre_c, ctx_dim); self.ctxp = nn.Linear(ctx_dim, ch)
        self.blocks = nn.ModuleList([Block(ch, d, ksize, drop) for d in dilations])
        self.use_attn = use_attn
        if use_attn:
            self.attn = SelfAttn(ch, attn_heads, drop)
            self.aln = nn.LayerNorm(ch)
        self.out = nn.Conv1d(ch, 5 if head == "mdn2" else 1, 1)
        self.mtp = nn.Conv1d(ch, len(MTP_OFFS), 1) if use_mtp else None
    def forward(self, x, m, px=None, pm=None):
        h = self.inp(self.indrop(x)) * m
        if self.use_ctx and px is not None:
            ctx = self.pre(px, pm)
            h = (h + self.ctxp(ctx)[:, :, None]) * m
        for b in self.blocks: h = b(h, m)
        if self.use_attn:
            q = self.aln(h.transpose(1, 2))
            a = self.attn(q, (m[:, 0, :] > 0.5))
            h = (h + a.transpose(1, 2)) * m
        mo = self.mtp(h) if self.mtp is not None else None
        return self.out(h), mo

def head_loss(out, yb, mb, head):
    if head == "mse":
        p = out[:, 0]
        return ((p - yb) ** 2 * mb).sum() / mb.sum(), p
    mu1, mu2 = out[:, 0], out[:, 1]
    pl = torch.sigmoid(out[:, 2])
    ls1 = out[:, 3].clamp(-3.0, 2.0); ls2 = out[:, 4].clamp(-3.0, 2.0)
    def logN(mu, ls): return -0.5 * ((yb - mu) / ls.exp()) ** 2 - ls - 0.5 * math.log(2 * math.pi)
    nll = -torch.logsumexp(torch.stack([torch.log(pl + 1e-6) + logN(mu1, ls1),
                                        torch.log(1 - pl + 1e-6) + logN(mu2, ls2)]), dim=0)
    mean = pl * mu1 + (1 - pl) * mu2
    loss = (nll * mb).sum() / mb.sum() + 0.1 * ((mean - yb) ** 2 * mb).sum() / mb.sum()
    return loss, mean

def predict_head(out, head):
    if head == "mse": return out[:, 0]
    pl = torch.sigmoid(out[:, 2]); return pl * out[:, 0] + (1 - pl) * out[:, 1]

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow: self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
    def copy_to(self, model):
        sd = model.state_dict()
        for k, v in self.shadow.items(): sd[k].copy_(v)

def make_batch(wids, train, rng, flip_all=False):
    """wids(井戸idリスト)→ (x, y, mask, sel, px, pm)。flip_all=True は TTA 用に全井戸反転。"""
    sel = []
    for w in wids:
        r = well_rows[w]
        if flip_all or (train and CFG["flip"] and rng.random() < 0.5): r = r[::-1]
        if train and len(r) > CROP:
            s = int(rng.integers(0, len(r) - CROP + 1)); r = r[s:s + CROP]
        sel.append((w, r))
    Lm = max(len(r) for _, r in sel); B = len(sel)
    xb = np.zeros((B, Lm, K_IN), np.float32)
    yb = np.zeros((B, Lm), np.float32); mb = np.zeros((B, Lm), np.float32)
    for i, (w, r) in enumerate(sel):
        xb[i, :len(r)] = X[r]; yb[i, :len(r)] = yn[r]; mb[i, :len(r)] = 1.0
    x = torch.from_numpy(xb).to(DEVICE).transpose(1, 2)
    yb = torch.from_numpy(yb).to(DEVICE); mb = torch.from_numpy(mb).to(DEVICE)
    px = pm = None
    if CFG["prefix"]:
        Lp = max(len(PREF.get(w, np.zeros((1, PRE_C), np.float32))) for w, _ in sel)
        pxb = np.zeros((B, Lp, PRE_C), np.float32); pmb = np.zeros((B, Lp), np.float32)
        for i, (w, _) in enumerate(sel):
            a = PREF.get(w)
            if a is not None: pxb[i, :len(a)] = a; pmb[i, :len(a)] = 1.0
        px = torch.from_numpy(pxb).to(DEVICE).transpose(1, 2)
        pm = torch.from_numpy(pmb).to(DEVICE).unsqueeze(1)
    return x, yb, mb, sel, px, pm

@torch.no_grad()
def _predict_once(model, head, wids, flip):
    model.eval(); out = np.zeros(N, np.float64)
    order = sorted(wids, key=lambda w: len(well_rows[w]))
    i = 0
    while i < len(order):
        Lm = len(well_rows[order[i]]); j = i + 1
        while j < len(order):
            Lc = max(Lm, len(well_rows[order[j]]))
            if (j - i + 1) * Lc > EVAL_TOK: break
            Lm = Lc; j += 1
        x, _, mb, sel, px, pm = make_batch(order[i:j], False, None, flip_all=flip)
        with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=(DEVICE == "cuda")):
            o = model(x, mb.unsqueeze(1), px, pm)[0]
        p = predict_head(o.float(), head).cpu().numpy()
        for b, (w, r) in enumerate(sel): out[r] = p[b, :len(r)]   # r 反転でも scatter で正位置
        i = j
    return out

def predict_wells(model, head, wids):
    p = _predict_once(model, head, wids, False)
    if CFG["flip"]:
        p = 0.5 * (p + _predict_once(model, head, wids, True))   # TTA: 正順+逆順平均
    return p


In [15]:
# ===== 学習(RUN プリセット): mdn2 × SEEDS × 5fold =====
def cosine_lr(step, total, warm):
    if step < warm: return step / max(1, warm)
    t = (step - warm) / max(1, total - warm)
    return 0.5 * (1.0 + math.cos(math.pi * t))

def new_model(head):
    return SeqNet(K_IN, head, CH, KSIZE, DILATIONS, DROP, USE_ATTN, ATTN_HEADS, CH_DROP,
                  use_ctx=CFG["prefix"], pre_c=PRE_C, ctx_dim=CTX_DIM,
                  use_mtp=CFG.get("mtp", False)).to(DEVICE)

def train_fold(head, seed, k):
    trw, vaw = fold_wells[k]
    fp = ART / "nn" / f"{RUN}_{head}_s{seed}_fold{k}.pt"
    model = new_model(head)
    if fp.exists():
        model.load_state_dict(torch.load(fp, map_location=DEVICE))
        return predict_wells(model, head, list(vaw)), "load"
    torch.manual_seed(1000 * seed + k); rng = np.random.default_rng(1000 * seed + k)
    trw = list(trw); vaw = list(vaw)
    psw = [w for w in PS_WELLS if PS_FOLD.get(w, -1) != k] if PSEUDO_MODE != "off" else []
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    n_ep_wells = len(trw) + (min(len(psw), int(len(trw) * DA_RATIO)) if PSEUDO_MODE == "da" else 0)
    steps_ep = math.ceil(n_ep_wells / BATCH_WELLS)
    total = steps_ep * EPOCHS; warm = int(total * WARM_FRAC)
    ema = EMA(model, EMA_DECAY); best = (1e18, None); step = 0
    va_idx = np.concatenate([well_rows[w] for w in vaw])
    if PSEUDO_MODE == "pretrain" and psw:                      # ===== pseudo事前学習(flat LR) =====
        pre_opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        for _pep in range(PRE_EPOCHS):
            model.train(); _po = rng.permutation(len(psw))
            for _pi in range(0, len(_po), BATCH_WELLS):
                x, yb, mb, _, px, pm = make_batch([psw[_pj] for _pj in _po[_pi:_pi + BATCH_WELLS]], True, rng)
                if NOISE: x = x + NOISE * torch.randn_like(x)
                with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=(DEVICE == "cuda")):
                    out, mo = model(x, mb.unsqueeze(1), px, pm)
                loss, _ = head_loss(out.float(), yb, mb, head)
                pre_opt.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
                pre_opt.step()
        ema = EMA(model, EMA_DECAY)                            # 事前学習済み重みでEMA再初期化
    for ep in range(EPOCHS):
        model.train()
        ep_wells = list(trw)
        if PSEUDO_MODE == "da" and psw:                        # ===== DA: pseudo井戸をエポック毎に混入 =====
            _nps = min(len(psw), int(len(trw) * DA_RATIO))
            ep_wells += [psw[_j] for _j in rng.choice(len(psw), size=_nps, replace=False)]
        order = rng.permutation(len(ep_wells))
        for i in range(0, len(order), BATCH_WELLS):
            x, yb, mb, _, px, pm = make_batch([ep_wells[j] for j in order[i:i + BATCH_WELLS]], True, rng)
            if NOISE: x = x + NOISE * torch.randn_like(x)
            for pg in opt.param_groups: pg["lr"] = LR * cosine_lr(step, total, warm)
            with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=(DEVICE == "cuda")):
                out, mo = model(x, mb.unsqueeze(1), px, pm)
            loss, _ = head_loss(out.float(), yb, mb, head)
            if CFG.get("mtp", False) and mo is not None:
                for _oi, _kk in enumerate(MTP_OFFS):
                    if yb.shape[1] > _kk:
                        _dt = yb[:, _kk:] - yb[:, :-_kk]
                        _vm = mb[:, _kk:] * mb[:, :-_kk]
                        if _vm.sum() > 0:
                            _pk = mo[:, _oi, :-_kk].float()
                            loss = loss + MTP_W * ((_pk - _dt) ** 2 * _vm).sum() / _vm.sum()
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            opt.step(); ema.update(model); step += 1
        bak = copy.deepcopy(model.state_dict()); ema.copy_to(model)
        p = predict_wells(model, head, vaw)
        rm = float(np.sqrt(np.mean((p[va_idx] * t_scale + t_mean - y[va_idx]) ** 2)))
        if not np.isfinite(rm):
            raise RuntimeError(f"val RMSE が非有限 (ep={ep}): 入力/損失にNaN。prefix等の入力を確認")
        if rm < best[0]: best = (rm, copy.deepcopy(model.state_dict()))
        model.load_state_dict(bak)
    model.load_state_dict(best[1]); torch.save(best[1], fp)
    return predict_wells(model, head, vaw), f"fit best={best[0]:.4f}"

nn_oof = {}
for head in HEADS:
    for seed in SEEDS:
        key = f"{head}_s{seed}"; oof = np.zeros(N, np.float64); t0 = time.time()
        for k in range(N_SPLITS):
            p, tag = train_fold(head, seed, k)
            va = folds[k][1]; oof[va] = p[va]
            gc.collect()
            if DEVICE == "cuda": torch.cuda.empty_cache()
            print(f"[{RUN}] {key} fold{k} [{tag}] "
                  f"rmse={np.sqrt(np.mean((p[va] * t_scale + t_mean - y[va]) ** 2)):.4f}", flush=True)
        nn_oof[key] = oof * t_scale + t_mean
        print(f"== [{RUN}] {key} OOF raw rmse={np.sqrt(np.mean((nn_oof[key] - y) ** 2)):.4f}"
              f" ({time.time() - t0:.0f}s)")
np.savez(ART / f"nn_oof_{RUN}.npz", **nn_oof)


[combo_pretrain-p1] mdn2_s0 fold0 [load] rmse=6.4376
[combo_pretrain-p1] mdn2_s0 fold1 [load] rmse=6.7346
[combo_pretrain-p1] mdn2_s0 fold2 [load] rmse=5.8784
[combo_pretrain-p1] mdn2_s0 fold3 [load] rmse=6.3735
[combo_pretrain-p1] mdn2_s0 fold4 [load] rmse=7.1846
== [combo_pretrain-p1] mdn2_s0 OOF raw rmse=6.5359 (8s)
[combo_pretrain-p1] mdn2_s1 fold0 [fit best=6.4660] rmse=6.4660
[combo_pretrain-p1] mdn2_s1 fold1 [fit best=6.5417] rmse=6.5417
[combo_pretrain-p1] mdn2_s1 fold2 [fit best=5.7304] rmse=5.7304
[combo_pretrain-p1] mdn2_s1 fold3 [fit best=6.2295] rmse=6.2295
[combo_pretrain-p1] mdn2_s1 fold4 [fit best=7.1033] rmse=7.1033
== [combo_pretrain-p1] mdn2_s1 OOF raw rmse=6.4297 (831s)
[combo_pretrain-p1] mdn2_s2 fold0 [fit best=6.6557] rmse=6.6557
[combo_pretrain-p1] mdn2_s2 fold1 [fit best=6.6550] rmse=6.6550
[combo_pretrain-p1] mdn2_s2 fold2 [fit best=5.8413] rmse=5.8413
[combo_pretrain-p1] mdn2_s2 fold3 [fit best=6.4750] rmse=6.4750
[combo_pretrain-p1] mdn2_s2 fold4 [fit best=7

In [16]:
# ===== 評価 + blend探索(vs v47 tabular GBM) + results_nn.json 蓄積 =====
pf_res = pf_abs - lk
w_pf = float(pp_params.get("w_pf", 0.09)); tau = pp_params.get("tau", 85)
alpha = float(pp_params.get("alpha", 1.0))
def apply_pp_res(res):
    d = res * (1 - w_pf) + pf_res * w_pf
    if tau: d = d * (1.0 - np.exp(-np.maximum(mds, 0.0) / float(tau)))
    return d * alpha
def raw_rmse(res): return float(np.sqrt(np.mean((res - y) ** 2)))
def overall(res):  return float(np.sqrt(np.mean((apply_pp_res(res) - y) ** 2)))
def wells_over(res, thr=10.0):
    e = apply_pp_res(res) - y
    return int(sum(1 for w in wid_order if np.sqrt(np.mean(e[well_rows[w]] ** 2)) > thr))

ens = np.mean(list(nn_oof.values()), axis=0)
ws = np.arange(0.0, 1.001, 0.02)
scs = [overall((1 - w) * gbm_oof + w * ens) for w in ws]
bi = int(np.argmin(scs)); wb = float(ws[bi])
res_b = (1 - wb) * gbm_oof + wb * ens

rec = dict(cfg=CFG, crop=CROP, seeds=len(SEEDS),
           nn_raw=raw_rmse(ens), nn_pp=overall(ens), nn_over10=wells_over(ens),
           blend_w=wb, blend_pp=float(scs[bi]), blend_over10=wells_over(res_b))
res_fp = ART / "results_nn.json"
allres = json.loads(res_fp.read_text()) if res_fp.exists() else {}
allres[RUN] = rec
res_fp.write_text(json.dumps(allres, indent=1))

print(f"[{RUN}] NN単体: raw={rec['nn_raw']:.4f} pp={rec['nn_pp']:.4f} >10={rec['nn_over10']}")
print(f"[{RUN}] blend : w={wb:.2f} pp={rec['blend_pp']:.4f} >10={rec['blend_over10']}"
      f"  (v47 GBM単体 {overall(gbm_oof):.4f})")
print()
print("=== 全プリセット比較(results_nn.json) ===")
print("参考 v29b(v28特徴): combo 単体pp=6.9403 blend=6.8711(w.62) / full blend=6.8768")
hdr = f"{'RUN':10s} {'nn_raw':>8s} {'nn_pp':>8s} {'>10':>4s} {'w':>5s} {'blend_pp':>9s} {'>10':>4s}  cfg"
print(hdr); print("-" * len(hdr))
for name, r in allres.items():
    on = "+".join(k for k, v in r["cfg"].items() if v) or "none"
    print(f"{name:10s} {r['nn_raw']:8.4f} {r['nn_pp']:8.4f} {r['nn_over10']:4d}"
          f" {r['blend_w']:5.2f} {r['blend_pp']:9.4f} {r['blend_over10']:4d}  {on} crop{r['crop']}")


[combo_pretrain-p1] NN単体: raw=6.4250 pp=6.4328 >10=66
[combo_pretrain-p1] blend : w=0.90 pp=6.4274 >10=65  (v47 GBM単体 6.7767)

=== 全プリセット比較(results_nn.json) ===
参考 v29b(v28特徴): combo 単体pp=6.9403 blend=6.8711(w.62) / full blend=6.8768
RUN          nn_raw    nn_pp  >10     w  blend_pp  >10  cfg
-----------------------------------------------------------
combo        6.4543   6.4573   66  0.84    6.4467   66  flip+prefix crop1024
combo_pretrain-p1   6.4250   6.4328   66  0.90    6.4274   65  flip+prefix crop1024


In [18]:
# ===== submit用パック書き出し(→ submit_pack_nn.json, submit_nn_v47 が読む) =====
EXPORT_RUN = "combo_pretrain-p1"          # ← 提出したい RUN 名(results_nn.json に存在すること)
_res = json.loads((ART / "results_nn.json").read_text())[EXPORT_RUN]
_cfg = _res["cfg"]
gmed = giqr = None
if _cfg.get("prefix"):        # prefix GR 正規化統計(抽出時と同一手順で train CSV から再計算)
    _grs = []
    for _w in wid_order:
        _f = DATA / "train" / f"{_w}__horizontal_well.csv"
        if not _f.exists(): continue
        _dfp = pd.read_csv(_f, usecols=["MD", "Z", "GR", "TVT_input"])
        _kn = _dfp.dropna(subset=["MD", "Z", "GR", "TVT_input"]).tail(PRE_LEN)
        if len(_kn) < 8: continue
        _grs.append(_kn["GR"].to_numpy(np.float64))
    _all = np.concatenate(_grs)
    gmed = float(np.nanmedian(_all))
    giqr = float(max(np.nanpercentile(_all, 75) - np.nanpercentile(_all, 25), 1e-6))
pack = dict(run=EXPORT_RUN, cfg=_cfg, w_nn=_res["blend_w"], oof_blend_pp=_res["blend_pp"],
            seeds=SEEDS, n_splits=N_SPLITS,
            features=feats_nn, abs2res=abs2res, med=med.tolist(), scale=scl.tolist(),
            t_mean=t_mean, t_scale=t_scale,
            arch=dict(ch=CH, ksize=KSIZE, dilations=DILATIONS, drop=DROP, use_attn=USE_ATTN,
                      attn_heads=ATTN_HEADS, ch_drop=CH_DROP,
                      pre_len=PRE_LEN, pre_c=PRE_C, ctx_dim=CTX_DIM, mtp_offs=MTP_OFFS),
            prefix_stats=dict(gmed=gmed, giqr=giqr))
(ART / "submit_pack_nn.json").write_text(json.dumps(pack))
print(f"submit_pack_nn.json saved: run={EXPORT_RUN} w_nn={_res['blend_w']}"
      f" oof_pp={_res['blend_pp']:.4f} prefix_stats=({gmed}, {giqr})")


submit_pack_nn.json saved: run=combo_pretrain-p1 w_nn=0.9 oof_pp=6.4274 prefix_stats=(83.38448782396254, 27.21518377216381)
